In [ ]:
# Inbound == False indicates outbound replies from brands
brand_tweets = df[df['inbound'] == False]
brand_counts = brand_tweets['author_id'].value_counts().head(20)

plt.figure(figsize=(12, 6))
brand_counts.plot(kind='bar')
plt.title('Top 20 Brands by Support Replies')
plt.xlabel('Brand')
plt.ylabel('Number of Replies')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(brand_counts)

In [ ]:
# Check substantive reply depth vs simple "DM us" loops
top_brands = brand_counts.head(5).index.tolist()

for brand in top_brands:
    print(f"\n{'='*60}\nBRAND: {brand}\n{'='*60}")
    brand_df = df[(df['author_id'] == brand) & (df['inbound'] == False)]
    samples = brand_df['text'].sample(5, random_state=42)
    for i, text in enumerate(samples):
        print(f"\n  Reply {i+1}: {text[:180]}...")
    avg_len = brand_df['text'].str.len().mean()
    print(f"\n  Average reply length: {avg_len:.1f} characters")

In [ ]:
BRAND = "AppleSupport"

brand_outbound = df[(df['author_id'] == BRAND) & (df['inbound'] == False)]
print(f"{BRAND} has {len(brand_outbound):,} outbound replies")

customer_tweet_ids = brand_outbound['in_response_to_tweet_id'].dropna()
brand_inbound = df[df['tweet_id'].isin(customer_tweet_ids)]
print(f"Corresponding customer messages: {len(brand_inbound):,}")

In [ ]:
import re
from collections import Counter

customer_msgs = brand_inbound['text'].dropna().tolist()

all_words = []
for msg in customer_msgs:
    clean = re.sub(r'@\w+', '', msg)
    clean = re.sub(r'http\S+', '', clean)
    words = clean.lower().split()
    all_words.extend([w for w in words if len(w) > 3])

word_freq = Counter(all_words)
print("Top 30 most frequent words in customer messages:")
for word, count in word_freq.most_common(30):
    print(f"  {word}: {count}")

In [ ]:
pairs = brand_outbound[['in_response_to_tweet_id', 'text']].copy()
pairs.columns = ['customer_tweet_id', 'brand_reply']
pairs = pairs.dropna(subset=['customer_tweet_id'])

pairs = pairs.merge(
    df[['tweet_id', 'text']].rename(columns={'text': 'customer_message'}),
    left_on='customer_tweet_id',
    right_on='tweet_id',
    how='inner'
)

print("="*80)
print("SAMPLE CONVERSATION PAIRS (Customer -> Brand)")
print("="*80)

samples = pairs.sample(10, random_state=42)
for idx, row in samples.iterrows():
    print(f"\n🧑 Customer: {row['customer_message']}")
    print(f"🤖 Brand:    {row['brand_reply']}")
    print("-" * 60)

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Save filtered tweets to avoid reloading the ~400MB CSV in future steps
brand_all = df[(df['author_id'] == BRAND) | (df['tweet_id'].isin(customer_tweet_ids))]
brand_all.to_csv(f'../data/processed/{BRAND}_all_tweets.csv', index=False)
print(f"Saved {len(brand_all):,} tweets for {BRAND}")

# Save pairs
pairs.to_csv(f'../data/processed/{BRAND}_pairs.csv', index=False)
print(f"Saved {len(pairs):,} conversation pairs")

In [ ]:
print("Data Types:\n", df.dtypes)
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nInbound distribution:\n{df['inbound'].value_counts()}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the raw dataset
df = pd.read_csv('../data/raw/twcs.csv')

print(f"Total rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print("\nFirst 5 rows:")
df.head()